<a href="https://colab.research.google.com/github/IsaacVic-Dark/Gemma-FineTune-colab-notebooks/blob/main/Load_InsuOps_Model_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — Install Unsloth stack (fresh runtime)

%%capture
import os

# Clean up conflicting packages
!pip uninstall -y torchao
!pip uninstall -y unsloth unsloth_zoo peft bitsandbytes accelerate xformers trl cut_cross_entropy

# Colab-specific Unsloth installation
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

In [ ]:
# !pip install torchao>=0.16.0

In [ ]:
# Cell 2 — Mount Google Drive and locate adapters

from google.colab import drive
drive.mount('/content/drive')

# Point to your adapter folder (update this path)
ADAPTER_DIR = "/content/drive/MyDrive/Insuops_lora_model_v2"

import os
print("Adapter dir exists:", os.path.isdir(ADAPTER_DIR))
print("Files:", os.listdir(ADAPTER_DIR))

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Cell 3 — Load base Gemma 3 model

from unsloth import FastModel

BASE_MODEL_NAME = "unsloth/gemma-3-1b-it"

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    dtype=None,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
    max_seq_length=2048,
)

print("Base model and tokenizer loaded.")

In [ ]:
# Cell 4 — Attach your LoRA adapters

# Load the LoRA adapter onto the base model
model.load_adapter(ADAPTER_DIR)

print("Adapter loaded.")

In [ ]:
# Cell 5 — Switch to inference mode (optional but recommended)

model = FastModel.for_inference(model)

print("Model set to inference mode.")

In [ ]:
# Cell 6 — Define a helper for prompting and test with a sample query

import torch

def format_chat_prompt(question: str) -> str:
    return (
        "<start_of_turn>user\n "
        f"{question}"
        "<end_of_turn><start_of_turn>model\n "
    )

def generate_answer(question: str, max_new_tokens: int = 256):
    prompt = format_chat_prompt(question)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    # Extract only the model turn after your prompt
    return full_text[len(prompt):]

question = "Explain how your quotation system calculates premiums."
print(generate_answer(question))